# Lab 26-04 · License or Build, Which Node, Chiplets or Not, Custom or Merchant

Companion to chapter 26-04. Each part starts with a prediction: write yours down before running the cell.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sourcing as s

## Part 1 · License or build

*Predict:* at what in-house NRE would building Example 26-04.2's PHY tie with licensing it at the planned volume, keeping the 25% chance of a $15 M slip?

In [2]:
units = [3, 5, 5, 3]                      # million units, years 1-4
U = s.discounted_units(units, 0.12)
lic = s.license_pv(6.0, 0.80, units, 0.12)
build = s.build_pv(24.0, 0.25, 15.0)
tie_nre = lic - 0.25 * 15.0
print(f"U = {U:.2f} M units, licence PV = ${lic:.2f} M, build PV = ${build:.2f} M")
print(f"build NRE that ties at plan volume: ${tie_nre:.2f} M")
u = np.linspace(0, 40, 201)
ustar = s.breakeven_discounted_units(build, 6.0, 0.80)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(u, 6.0 + 0.80 * u, label="license: F + rU")
ax.plot(u, np.full_like(u, build), label="build: N + p C")
ax.axvline(U, ls=":", color="k"); ax.axvline(ustar, ls="--", color="gray")
ax.set_xlabel("discounted unit stream U (million)"); ax.set_ylabel("present cost ($M)"); ax.legend()
print(f"break-even U* = {ustar:.1f} M, {ustar / U:.2f} x plan")

U = 12.13 M units, licence PV = $15.70 M, build PV = $27.75 M
build NRE that ties at plan volume: $11.95 M
break-even U* = 27.2 M, 2.24 x plan


## Part 2 · Amdahl's law for area

*Predict:* what logic gain does a 50/30/20 die need for an overall 1.5×, with SRAM at 1.2× and analog at 1.1×?

In [3]:
fr, gains = [0.5, 0.3, 0.2], [1.2, 1.1]
rest = fr[1] / gains[0] + fr[2] / gains[1]
need = fr[0] / (1 / 1.5 - rest)
print(f"SRAM + analog terms = {rest:.4f}; logic gain needed = {need:.2f}")
sl = np.linspace(1, 10, 181)
seff = [s.effective_density_gain(fr, [x] + gains) for x in sl]
for x in (3, 10):
    print(f"logic {x}x -> overall {s.effective_density_gain(fr, [x] + gains):.2f}x")
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(sl, seff); ax.axhline(1 / rest, ls="--", color="gray", label=f"ceiling {1 / rest:.2f}")
ax.set_xlabel("logic density gain"); ax.set_ylabel("overall density gain"); ax.legend()

SRAM + analog terms = 0.4318; logic gain needed = 2.13
logic 3x -> overall 1.67x
logic 10x -> overall 2.08x


## Part 3 · Chiplet family

*Predict:* with L fixed at 1 million units, how large must S's volume grow before the monolithic family becomes cheaper?

In [4]:
ms = s.good_die_cost(20000, 200, 0.10, test=2)[2]
ml = s.good_die_cost(20000, 350, 0.10, test=2)[2]
cc = s.good_die_cost(20000, 165, 0.10, test=2)[2]
io = s.good_die_cost(10000, 55, 0.08, test=2)[2]
S_m, L_m = s.package_cost([ms], 15, 0.99), s.package_cost([ml], 15, 0.99)
S_c, L_c = s.package_cost([cc, io], 25, 0.99), s.package_cost([cc, cc, io], 35, 0.99)
vs = np.linspace(0, 25, 251)
mono = [s.family_total(190, [S_m, L_m], [v, 1]) for v in vs]
chip = [s.family_total(126, [S_c, L_c], [v, 1]) for v in vs]
flip = (64 + (L_m - L_c)) / (S_c - S_m)  # 190 + V S_m + L_m = 126 + V S_c + L_c
print(f"S unit penalty ${S_c - S_m:.2f}, L unit saving ${L_m - L_c:.2f}; families tie at S = {flip:.1f} M units")
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(vs, mono, label="monolithic"); ax.plot(vs, chip, label="chiplets")
ax.set_xlabel("S lifetime volume (million)"); ax.set_ylabel("family total ($M)"); ax.legend()

S unit penalty $4.72, L unit saving $9.31; families tie at S = 15.5 M units


## Part 4 · Custom or merchant

*Predict:* at what merchant price does the custom accelerator save nothing per merchant-equivalent?

In [5]:
custom = s.tco_per_throughput(9000, 0.7, 0.8, 4, 1.3, 0.08, 12000)
energy_m = 1.0 * 1.3 * s.HOURS_PER_YEAR * 4 * 0.08
floor = custom - 12000 - energy_m
print(f"custom cost per merchant-equivalent ${custom:,.0f}; no saving below a merchant price of ${floor:,.0f}")
prices = np.linspace(15000, 40000, 251)
fleet = [800e6 / (s.tco_per_throughput(p, 1.0, 1.0, 4, 1.3, 0.08, 12000) - custom) / 1e3 for p in prices]
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(prices / 1e3, fleet); ax.set_yscale("log")
ax.set_xlabel("merchant price ($k)"); ax.set_ylabel("break-even fleet (k merchant-equivalents)")

custom cost per merchant-equivalent $29,439; no saving below a merchant price of $13,794


Text(0, 0.5, 'break-even fleet (k merchant-equivalents)')